# Notebook 3 — Analyse des résultats (CPU, sans quota GPU)

**Réglages :** *Accelerator* **None** ; *Internet* non nécessaire.
*Add Input* : le dataset du code **et** la sortie du (des) notebook(s) d'entraînement.
Si plusieurs runs sont présents (ex. v1 et v2), ils sont analysés et comparés.

Pour chaque run : adaptation aux prévalences (EM et BBSE, sans label externe), recalibration
few-shot (200 ECG étiquetés), seuil oracle, courbes de fiabilité, gating par accord
texte / sans texte, tests de DeLong appariés et IC bootstrap.

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

def read_or_empty(path):
    """Lit un CSV d'analyse ; vide ou absent (ex. run « signal seul » sans texte) -> tableau vide."""
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()
runs = sorted({os.path.dirname(os.path.dirname(p))
               for p in glob.glob("/kaggle/input/**/runs/*/eval/summary.csv", recursive=True)})
runs = [r for r in runs if not r.endswith("/test")]
assert runs, "Aucun run trouvé : ajoutez la sortie du notebook d'entraînement en Input."
for r in runs:
    print(" -", r)

In [ ]:
OUT = "/kaggle/working/analysis"
for r in runs:
    name = os.path.basename(r)
    print("=" * 30, name, "=" * 30)
    !python scripts/analyze_preds.py --run {r} --out {OUT}/{name}

## Comparaison des runs (AUROC macro)

In [ ]:
tab = []
for r in runs:
    s = pd.read_csv(f"{r}/eval/summary.csv")
    s["run"] = os.path.basename(r)
    tab.append(s)
tab = pd.concat(tab)
keep = ["full", "no_text", "signal_only", "leadI_no_text", "text_only"]
display(tab[tab.scenario.isin(keep)].pivot_table(index=["eval_set", "scenario"], columns="run",
        values="macro_auroc").round(3))

## Seuils et prévalences : brut vs EM / BBSE vs few-shot vs oracle

In [ ]:
for r in runs:
    name = os.path.basename(r)
    d = read_or_empty(f"{OUT}/{name}/label_shift_summary.csv")
    print(name)
    if d.empty or "scenario" not in d.columns:
        print("  (non applicable pour ce run)"); continue
    cols = [c for c in ["eval_set", "scenario", "prev_true", "prev_em", "prev_bbse", "sens_raw", "sens_em",
                        "sens_bbse", "sens_fewshot", "sens_oracle", "f1_raw", "f1_em", "f1_bbse",
                        "f1_fewshot", "f1_oracle", "ece_raw", "ece_platt"] if c in d.columns]
    display(d[d.scenario.isin(["full", "no_text"])][cols].round(3))

## Gating par accord et tests statistiques

In [ ]:
for r in runs:
    name = os.path.basename(r)
    print(name)
    for f in ["gating.csv", "delta_macro_auroc_bootstrap.csv"]:
        t = read_or_empty(f"{OUT}/{name}/{f}")
        display(t.round(4)) if not t.empty else print(f"  {f} : non applicable pour ce run")

## Figures de calibration pour l'article (externes ; PNG et PDF dans analysis/<run>/)

In [ ]:
for r in runs:
    name = os.path.basename(r)
    for f in sorted(glob.glob(f"{OUT}/{name}/fig_calibration_*.png")):
        print(name, os.path.basename(f)); display(Image(f))